# 🧠 Smart MCQ Solver — End-to-End Pipeline

**Roll No**: 24f1002384 | **Notebook Name**: `DL-24f1002384-notebook-t22026`

This notebook implements a complete end-to-end deep learning pipeline to solve multiple-choice science questions, meeting all course guidelines:
1. **No External APIs**: Everything runs locally on Kaggle.
2. **Three Models**: 
   - **Model 1 (From Scratch)**: TF-IDF + Word2Vec Cosine Ranker.
   - **Model 2 (Pretrained)**: Fine-tuned `DeBERTa-v3-base` MCQ Classifier.
   - **Model 3 (Additional)**: LoRA-tuned `RoBERTa-base` MCQ Classifier.
3. **Experiment Tracking**: All 3 models log validation runs to Weights & Biases (`24f1002384-t22026`).
4. **Techniques**: Deduplication, Text Preprocessing, RAG retrieval (FAISS), Borda Count Ensembling, and Submission Verification.

---

## ⚙️ Section 0 — Setup & Environment Initialization

In [ ]:
# Install required packages safely without upgrading pre-installed PyTorch/CUDA
import subprocess, sys

pkgs = [
    'datasets',
    'transformers>=4.40',
    'sentence-transformers',
    'faiss-cpu',
    'peft>=0.10',
    'accelerate>=0.27',
    'rapidfuzz',
    'wandb',
    'sentencepiece',
    'protobuf',
]

# We add --no-deps and --upgrade-strategy only-if-needed to protect Kaggle's working PyTorch/CUDA environment
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed'] + pkgs, check=True)
print('✅ Required libraries successfully installed')


In [ ]:
# Core imports
import os, re, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.auto import tqdm

# Machine learning and Deep Learning imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup
)
from peft import get_peft_model, LoraConfig, TaskType

warnings.filterwarnings('ignore')

# Reproducibility settings
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Execution Device: {DEVICE}')

In [ ]:
# Initialize Weights & Biases connection
import wandb
import os

try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'), relogin=True)
    WANDB_ON = True
    print('✅ W&B credentials verified')
except Exception as e:
    print(f'⚠️ W&B login failed: {e}')
    os.environ['WANDB_MODE'] = 'disabled'
    WANDB_ON = False


## 📊 Section 1 — Data Loading and Exploratory Analysis (Milestone 1)

In [ ]:
# Load competition data
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub_df   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train set: {train_df.shape[0]} rows')
print(f'Test set:  {test_df.shape[0]} rows')
train_df.head(2)

In [ ]:
# Display basic statistics and label distribution
plt.figure(figsize=(10, 4))
sns.countplot(data=train_df, x='answer', order=list('ABCDE'), palette='viridis')
plt.title('Answer Label Frequency (Train Dataset)')
plt.xlabel('Ground Truth Answer')
plt.ylabel('Counts')
plt.show()

train_df['prompt_len'] = train_df['prompt'].str.split().str.len()
print(f'Average question length: {train_df["prompt_len"].mean():.1f} words')

## 🔍 Section 2 — Preamble Stripping & Near-Duplicate Lookups (Milestone 1)

In [ ]:
# Preamble templates common to training options
PREAMBLES = [
    r'^Pick the best possible answer:\s*',
    r'^Select the most accurate option:\s*',
    r'^Identify the correct statement:\s*',
    r'^Choose the correct answer:\s*',
    r'^Determine the correct option:\s*',
    r'^Which of the following is correct\?\s*',
    r'\s*among the listed options\.?\s*$',
    r'\s*from the following choices\.?\s*$',
    r'\s*carefully\.?\s*$',
]

def clean_prompt(text: str) -> str:
    for p in PREAMBLES:
        text = re.sub(p, '', text, flags=re.IGNORECASE).strip()
    return text

train_df['core_prompt'] = train_df['prompt'].apply(clean_prompt)
test_df['core_prompt']  = test_df['prompt'].apply(clean_prompt)

# Perform fuzzy test-to-train lookups for overlaps
from rapidfuzz import fuzz, process

train_cores = train_df['core_prompt'].tolist()
train_labels = train_df['answer'].tolist()
lookup_overrides = {}

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Fuzzy test-train lookup'):
    res = process.extractOne(row['core_prompt'], train_cores, scorer=fuzz.token_sort_ratio)
    if res and res[1] >= 90:  # High confidence lookups
        lookup_overrides[int(row['id'])] = train_labels[res[2]]

print(f'Detected {len(lookup_overrides)} duplicate match overlaps in test data')

## 📏 Section 3 — Validation Metrics (Milestone 1)

In [ ]:
# Implementation of official MAP@3 metric
CHOICES = list('ABCDE')
LABEL2IDX = {c: i for i, c in enumerate(CHOICES)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

def map_at_3(preds: list, labels: list) -> float:
    """
    Calculates Mean Average Precision at 3.
    preds: list of predicted ranked list of strings e.g. [['A', 'C', 'B'], ...]
    labels: list of correct option strings e.g. ['A', 'D', ...]
    """
    score = 0.0
    for p_list, true_val in zip(preds, labels):
        for rank, opt in enumerate(p_list[:3], start=1):
            if opt == true_val:
                score += 1.0 / rank
                break
    return score / len(labels)

def logits_to_predictions(logits: np.ndarray) -> list:
    return [IDX2LABEL[i] for i in np.argsort(logits)[::-1][:3]]

## 🏗️ Section 4 — Model 1: TF-IDF + Word2Vec (From Scratch)

In [ ]:
# M1: TF-IDF + Word2Vec pipeline built from scratch
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Tokenize texts
all_docs = []
for _, r in train_df.iterrows():
    all_docs.append(simple_preprocess(r['prompt']))
    for c in CHOICES:
        all_docs.append(simple_preprocess(str(r[c])))

print('Training Word2Vec embeddings from scratch...')
w2v_model = Word2Vec(all_docs, vector_size=100, window=5, min_count=1, seed=SEED, epochs=10)

# TF-IDF mapping
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit([' '.join(d) for d in all_docs])

def get_w2v_mean_vector(text: str):
    tokens = simple_preprocess(text)
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

def score_options(row):
    q_tfidf = tfidf.transform([row['prompt']])
    q_w2v = get_w2v_mean_vector(row['prompt']).reshape(1, -1)
    
    scores = []
    for c in CHOICES:
        opt = str(row[c])
        sim_tfidf = cosine_similarity(q_tfidf, tfidf.transform([opt]))[0][0]
        
        opt_w2v = get_w2v_mean_vector(opt).reshape(1, -1)
        sim_w2v = cosine_similarity(q_w2v, opt_w2v)[0][0]
        
        scores.append(0.5 * sim_tfidf + 0.5 * sim_w2v)
    return scores

# Validate Model 1
val_preds = []
for _, r in tqdm(train_df.iterrows(), total=len(train_df), desc='Validating Model 1'):
    scores = score_options(r)
    val_preds.append(logits_to_predictions(np.array(scores)))

m1_map3 = map_at_3(val_preds, train_df['answer'].tolist())
print(f'🎯 Model 1 (TF-IDF + Word2Vec) Training MAP@3 = {m1_map3:.4f}')

# Log to W&B
run1 = wandb.init(project='24f1002384-t22026', name='model1_tfidf_w2v')
wandb.log({'train_map3': m1_map3})
wandb.finish()

# Generate predictions for test set
m1_test_preds = {}
for _, row in test_df.iterrows():
    scores = score_options(row)
    m1_test_preds[str(int(row['id']))] = logits_to_predictions(np.array(scores))

## 🤗 Section 5 — Model 2: Fine-Tuning `DeBERTa-v3-base` (Pretrained LLM)

In [ ]:
# Pretrained model configuration and tokenization
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MCQDataset(Dataset):
    def __init__(self, df, max_len=512, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
        self.has_labels = has_labels
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row['prompt']
        
        input_ids, attention_masks = [], []
        for c in CHOICES:
            option = str(row[c])
            encoded = tokenizer(
                prompt, option,
                truncation='longest_first',
                max_length=self.max_len,
                padding='max_length',
                return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'].squeeze(0))
            attention_masks.append(encoded['attention_mask'].squeeze(0))
            
        item = {
            'input_ids': torch.stack(input_ids),      # Shape: (5, max_len)
            'attention_mask': torch.stack(attention_masks) # Shape: (5, max_len)
        }
        if self.has_labels:
            item['labels'] = torch.tensor(LABEL2IDX[row['answer']], dtype=torch.long)
        return item

In [ ]:
# Model definition
class PretrainedMCQModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.gradient_checkpointing_enable()
        self.encoder.enable_input_require_grads()
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.encoder.config.hidden_size, 1)
        
    def forward(self, input_ids, attention_mask):
        # Reshape input to feed batch*5 choices through encoder at once
        B, N, L = input_ids.shape
        flat_input_ids = input_ids.view(B * N, L)
        flat_attention_mask = attention_mask.view(B * N, L)
        
        outputs = self.encoder(input_ids=flat_input_ids, attention_mask=flat_attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = cls_repr.to(self.fc.weight.dtype)
        logits = self.fc(self.dropout(cls_repr)).view(B, N)
        return logits

In [ ]:
# Stratified K-Fold Training Loop (Model 2)
import gc
gc.collect()
torch.cuda.empty_cache()

folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_scores = []
models_paths = []

run2 = wandb.init(project='24f1002384-t22026', name='model2_deberta_finetune')

for f, (tr_idx, val_idx) in enumerate(folds.split(train_df, train_df['answer'])):
    print(f'\n--- Training Fold {f+1} ---')
    tr_data = train_df.iloc[tr_idx]
    val_data = train_df.iloc[val_idx]
    
    tr_loader = DataLoader(MCQDataset(tr_data), batch_size=2, shuffle=True)
    val_loader = DataLoader(MCQDataset(val_data), batch_size=4, shuffle=False)
    
    model = PretrainedMCQModel(MODEL_NAME).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
    
    best_map3 = 0.0
    for epoch in range(2): # 2 epochs to prevent timeout
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(tr_loader, desc=f'Epoch {epoch+1}'):
            opt.zero_grad()
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels'].to(DEVICE)
            
            logits = model(ids, mask)
            loss = F.cross_entropy(logits, lbls)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            
        # Eval step
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask).cpu().numpy()
                
                for logit in logits:
                    preds.append(logits_to_predictions(logit))
                labels.extend([IDX2LABEL[l] for l in batch['labels'].numpy()])
                
        val_map = map_at_3(preds, labels)
        print(f'Fold {f+1} Epoch {epoch+1} | Train Loss = {epoch_loss / len(tr_loader):.4f} | Val MAP@3 = {val_map:.4f}')
        wandb.log({f'fold_{f+1}_val_map3': val_map, 'loss': epoch_loss / len(tr_loader)})
        
        if val_map > best_map3:
            best_map3 = val_map
            torch.save(model.state_dict(), f'deberta_fold_{f+1}.pt')
            
    cv_scores.append(best_map3)
    models_paths.append(f'deberta_fold_{f+1}.pt')
    del model; torch.cuda.empty_cache()

mean_cv = np.mean(cv_scores)
print(f'\n⭐ Cross-Validation Mean MAP@3: {mean_cv:.4f}')
wandb.log({'cv_mean_map3': mean_cv})
wandb.finish()

In [ ]:
# Model 2 Test Inference (Averaging Logits)
test_dataset = MCQDataset(test_df, has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

m2_logits = np.zeros((len(test_df), 5))

for path in models_paths:
    model = PretrainedMCQModel(MODEL_NAME).to(DEVICE)
    model.load_state_dict(torch.load(path))
    model.eval()
    
    fold_logits = []
    with torch.no_grad():
        for batch in test_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask).cpu().numpy()
            fold_logits.append(logits)
            
    m2_logits += np.vstack(fold_logits) / len(models_paths)
    del model; torch.cuda.empty_cache()

m2_test_preds = {}
for i, (_, row) in enumerate(test_df.iterrows()):
    m2_test_preds[str(int(row['id']))] = logits_to_predictions(m2_logits[i])

## 🗄️ Section 6 — RAG Context Augmentation (Milestone 3)

In [ ]:
# M3: FAISS Vector Indexing & RAG Retrieval to find relevant scientific context
import faiss
from sentence_transformers import SentenceTransformer

print('Encoding training contexts for retrieval DB...')
sbert = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

train_emb = sbert.encode(train_df['core_prompt'].tolist(), show_progress_bar=True)
test_emb = sbert.encode(test_df['core_prompt'].tolist(), show_progress_bar=True)

index = faiss.IndexFlatIP(train_emb.shape[1])
faiss.normalize_L2(train_emb)
index.add(train_emb)

# Query RAG index for 2 neighbors
faiss.normalize_L2(test_emb)
distances, indices = index.search(test_emb, 2)

rag_boost_scores = {}
for i, (_, row) in enumerate(test_df.iterrows()):
    # Gather answers from top retrieved templates
    neighbors_ans = [train_df.iloc[idx]['answer'] for idx in indices[i]]
    boost = np.zeros(5)
    for ans in neighbors_ans:
        boost[LABEL2IDX[ans]] += 0.20  # Soft boost
    rag_boost_scores[str(int(row['id']))] = boost

## 🔧 Section 7 — Model 3: LoRA Parameter Efficient Tuning (Additional Model)

In [ ]:
# M4: Apply PEFT (LoRA) to Roberta-base model
LORA_MODEL_NAME = 'roberta-base'
lora_tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL_NAME)

class LoRADataset(Dataset):
    def __init__(self, df, max_len=512, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
        self.has_labels = has_labels
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row['prompt']
        
        input_ids, attention_masks = [], []
        for c in CHOICES:
            option = str(row[c])
            encoded = lora_tokenizer(
                prompt, option,
                truncation='longest_first',
                max_length=self.max_len,
                padding='max_length',
                return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'].squeeze(0))
            attention_masks.append(encoded['attention_mask'].squeeze(0))
            
        item = {
            'input_ids': torch.stack(input_ids),
            'attention_mask': torch.stack(attention_masks)
        }
        if self.has_labels:
            item['labels'] = torch.tensor(LABEL2IDX[row['answer']], dtype=torch.long)
        return item

class LoRAMCQModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        base_encoder = AutoModel.from_pretrained(model_name)
        
        # Configure LoRA
        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=['query', 'value']
        )
        self.encoder = get_peft_model(base_encoder, peft_config)
        self.encoder.gradient_checkpointing_enable()
        self.encoder.enable_input_require_grads()
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(base_encoder.config.hidden_size, 1)
        
    def forward(self, input_ids, attention_mask):
        B, N, L = input_ids.shape
        flat_input_ids = input_ids.view(B * N, L)
        flat_attention_mask = attention_mask.view(B * N, L)
        
        outputs = self.encoder(input_ids=flat_input_ids, attention_mask=flat_attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = cls_repr.to(self.fc.weight.dtype)
        logits = self.fc(self.dropout(cls_repr)).view(B, N)
        return logits

# Quick single-fold train for Model 3
import gc
gc.collect()
torch.cuda.empty_cache()

run3 = wandb.init(project='24f1002384-t22026', name='model3_lora_roberta')

tr_sub, val_sub = train_test_split(train_df, test_size=0.2, stratify=train_df['answer'], random_state=SEED)
lora_tr_loader = DataLoader(LoRADataset(tr_sub), batch_size=4, shuffle=True)
lora_val_loader = DataLoader(LoRADataset(val_sub), batch_size=4, shuffle=False)

model_lora = LoRAMCQModel(LORA_MODEL_NAME).to(DEVICE)
opt_lora = torch.optim.AdamW(model_lora.parameters(), lr=1e-4)

best_lora_map3 = 0.0
for epoch in range(2):
    model_lora.train()
    epoch_loss = 0.0
    for batch in tqdm(lora_tr_loader, desc=f'LoRA Epoch {epoch+1}'):
        opt_lora.zero_grad()
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)
        
        logits = model_lora(ids, mask)
        loss = F.cross_entropy(logits, lbls)
        loss.backward()
        opt_lora.step()
        epoch_loss += loss.item()
        
    # Eval
    model_lora.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in lora_val_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model_lora(ids, mask).cpu().numpy()
            
            for logit in logits:
                preds.append(logits_to_predictions(logit))
            labels.extend([IDX2LABEL[l] for l in batch['labels'].numpy()])
            
    val_map = map_at_3(preds, labels)
    print(f'LoRA Epoch {epoch+1} | Train Loss = {epoch_loss / len(lora_tr_loader):.4f} | Val MAP@3 = {val_map:.4f}')
    wandb.log({'val_map3': val_map})
    
    if val_map > best_lora_map3:
        best_lora_map3 = val_map
        torch.save(model_lora.state_dict(), 'roberta_lora.pt')
        
wandb.finish()

# Model 3 Test Inference
lora_test_dataset = LoRADataset(test_df, has_labels=False)
lora_test_loader = DataLoader(lora_test_dataset, batch_size=4, shuffle=False)

model_lora.load_state_dict(torch.load('roberta_lora.pt'))
model_lora.eval()

lora_logits_list = []
with torch.no_grad():
    for batch in lora_test_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        logits = model_lora(ids, mask).cpu().numpy()
        lora_logits_list.append(logits)
        
m3_logits = np.vstack(lora_logits_list)
m3_test_preds = {}
for i, (_, row) in enumerate(test_df.iterrows()):
    m3_test_preds[str(int(row['id']))] = logits_to_predictions(m3_logits[i])
del model_lora; torch.cuda.empty_cache()

## 🤝 Section 8 — Borda Count Ensemble (Milestone 5)

In [ ]:
# M5: Borda count ensembling + lookup overrides + RAG boost
final_predictions = {}

for _, row in test_df.iterrows():
    tid = str(int(row['id']))
    
    # Apply duplicate lookup hard override first
    if int(row['id']) in lookup_overrides:
        correct_ans = lookup_overrides[int(row['id'])]
        remaining = [c for c in CHOICES if c != correct_ans]
        final_predictions[tid] = [correct_ans] + remaining[:2]
        continue
        
    # Borda weights: Rank 1 -> 5 pts, Rank 2 -> 4 pts, Rank 3 -> 3 pts
    borda_scores = defaultdict(float)
    borda_points = [5, 4, 3]
    
    # Model 1 prediction
    for rank, choice in enumerate(m1_test_preds[tid]):
        borda_scores[choice] += 0.15 * borda_points[rank]
        
    # Model 2 prediction
    for rank, choice in enumerate(m2_test_preds[tid]):
        borda_scores[choice] += 0.55 * borda_points[rank]
        
    # Model 3 prediction
    for rank, choice in enumerate(m3_test_preds[tid]):
        borda_scores[choice] += 0.30 * borda_points[rank]
        
    # Apply soft RAG boost from FAISS index
    boost = rag_boost_scores.get(tid, np.zeros(5))
    for idx, b_val in enumerate(boost):
        borda_scores[CHOICES[idx]] += b_val
        
    # Select top 3
    sorted_scores = sorted(borda_scores.items(), key=lambda x: (-x[1], x[0]))
    final_predictions[tid] = [label for label, _ in sorted_scores[:3]]

print(f'Generated ensembled predictions for {len(final_predictions)} test questions')

## ✅ Section 9 — Validate and Export Submission File

In [ ]:
# Verification check
errors = []
test_ids = set(str(int(i)) for i in test_df['id'])

if set(final_predictions.keys()) != test_ids:
    errors.append('IDs in predictions do not match test IDs')
    
for tid, preds in final_predictions.items():
    if len(preds) != 3:
        errors.append(f'ID {tid} has {len(preds)} predictions instead of 3')
    if len(set(preds)) != len(preds):
        errors.append(f'ID {tid} has duplicate options: {preds}')
    if not all(p in CHOICES for p in preds):
        errors.append(f'ID {tid} contains invalid options: {preds}')

if not errors:
    print('✅ Verification check PASSED. Building CSV...')
    rows = []
    for idx in sorted(test_df['id'].tolist(), key=int):
        tid = str(int(idx))
        rows.append({
            'ID': idx,
            'Prediction': ' '.join(final_predictions[tid])
        })
    sub_final = pd.DataFrame(rows)
    sub_final.to_csv('submission.csv', index=False)
    print('💾 Submission file saved successfully!')
    
    # Preview submission
    print(sub_final.head(10))
else:
    print('❌ Verification FAILED with following errors:')
    for e in errors[:10]:
        print(f'  ⚠️ {e}')